# First steps with Jupyter notebboks

This is a [Jupyter notebok](https://jupyter.org/), the standard format for graphical interaction with [SageMath](https://www.sagemath.org/), but also popular for Python and other languages. This exercise session will be enitrely in Python, but if you installed SageMath you may use it to launch Jupyter.

You can download it to a folder on your computer. Then to open it launch SageMath's graphical user interface (in a terminal, type `sage -n`, or find the launch icon on your desktop) and navigate to the folder.

## Binary exponentiation modulo N

In this exercise we are going to implement in Python efficient exponentiation modulo an integer N.

We want to write a function `binexp(a, b, N)` that computes $a^b \mod N$. Here is a skeleton for the function, that only works for b = 0, 1.

In [5]:
def binexp(a, b, N):
  n = 1
  for i in range(b):
    n = (a*n) % N
  return n


1. Try the function on some inputs

In [ ]:
binexp(3, 0, 13)

1

2. Implement the cases b = 2, 3

3. Using a [`for` loop](https://docs.python.org/3/tutorial/controlflow.html#for-statements), make the function work for any positive b

4. Run the command below to measure the time spent computing $2^{10^8} \mod 13$. How much time it takes to raise to the $10^{9}$-th power? To the $10^{12}$-th?

In [10]:
%time binexp(2, 100000000, 13)

CPU times: user 8.8 s, sys: 2.64 ms, total: 8.8 s
Wall time: 8.89 s


3

5. Now write an **efficient** function for exponentiating. Below is a skeleton of a function that reads the exponent b bit-by-bit from right to left. Modify the funciton to compute $a^b \mod N$.

In [29]:
def binexp_fast(a, b, N):
    n = 1
    a2i = a
    while b > 0:
      if b % 2 != 0:
        n = (n * a2i) % N
      
      a2i = a2i * a2i % N
      b = b // 2
    return n


Try the function out

In [30]:
for j in range(2, 10):
  print("\n", j, end=":")
  for i in range(10):
    print(binexp_fast(j, i, 3000), end=",")


 2:1,2,4,8,16,32,64,128,256,512,
 3:1,3,9,27,81,243,729,2187,561,1683,
 4:1,4,16,64,256,1024,1096,1384,2536,1144,
 5:1,5,25,125,625,125,625,125,625,125,
 6:1,6,36,216,1296,1776,1656,936,2616,696,
 7:1,7,49,343,2401,1807,649,1543,1801,607,
 8:1,8,64,512,1096,2768,1144,152,1216,728,
 9:1,9,81,729,561,2049,441,969,2721,489,

6. Use the `%time` magic to compare to your previous version of `binexp`. Python already has the same function implemented as `pow(a, b, N)`, compare with that one too.

In [31]:
%time binexp_fast(2, 100000000, 200)

CPU times: user 113 μs, sys: 0 ns, total: 113 μs
Wall time: 139 μs


176

7. (Optional). If you're familiar with the bit operators `&`, `>>` etc., implement left-to-right binary exponentiation.

## Primality testing

We want to generate suitable primes for Diffie-Hellman. The usual way to find a prime in a given interval is to take a random number in that interval and test for primality, until a prime is found. By the prime number theorem we know that roughly $1/\log(N)$ integers in the $[0,N]$ interval are prime, thus we expect to need $≈ \log(N)$ trials.

The Fermat test is a *compositeness* test. Given an integer $N$, it:
- takes a random $a \in [2, N-1]$
- computes $b = a^{N-1} \mod N$
- returns *composite* if $b ≠ 1$, *unknown* otherwise.

If the Fermat test returns *composite*, then $N$ is certainly not prime, because of Fermat's little theorem. If it returns *unknown*, then we can't conclude $N$ is prime, however the more Miller-Rabin tests we run, the fewer the chances that $N$ is composite. Be wary of [Carmichael numbers](https://en.wikipedia.org/wiki/Carmichael_number), though.

Implement Fermat's compositeness test and use it to generate a probable prime of 60 binary digits.

If you want a more robust test, read about the [Miller-Rabin test](https://en.wikipedia.org/wiki/Miller%E2%80%93Rabin_primality_test).

In [ ]:
import random

def mrtest(N):
  if (N < 2):
    return False

  a = random.randint(1, N - 1)
  return binexp_fast(a, N - 1, N) == 1

In [106]:
import math

def isprime(N):
  for i in range(2, int(math.sqrt(N))):
    if N % i == 0:
      return False
  return True

for j in range(10000000):
  c = 0
  for i in range(100):
    if (mrtest(i) == isprime(i)):
      c = c + 1
    # print(i, " : ", mrtest(i) == isprime(i))

  if (c > 94):
    print(c)

95
95
95
96
96
95
95
96
95
95
95
95
95
95
95
96
95
95
95
95
95
95
95
95
95
95
95
95
95
95
95
95
95
95
95
95
95


KeyboardInterrupt: 

## Diffie-Hellman

Demonstrate the Diffie-Hellman key exchange in $ℤ/pℤ$, using the prime previously found.

## Baby-step giant-step

Implement the [baby-step giant-step](https://en.wikipedia.org/wiki/Baby-step_giant-step) algorithm for the discrete logarithm problem in $ℤ/pℤ$ and use it to break the Diffie-Hellman key exchange above.

As a reminder, the idea is to:
1. Pre-compute a table of all values $g^i$ for $1 ≤ i ≤ N$ with $N ≈ \sqrt{p}$ and $\gcd(N, p-1) = 1$;
2. To find the discrete logarithm of $h = g^x$, compute $h/g^{Nj}$ for $1 ≤ j ≤ \lceil p/N\rceil$ and look for a match in the table.

Python offers a data structure named `dict()` that implements tables with fast look-up.

# Pen-and-paper exercises

## 1. One-time pad is perfectly secure

A cipher is perfectly secure if $∀ m_0, m_1, c$ we have $Pr[Enc_k(m_0) = c] = Pr[Enc_k(m_1) = c]$, assuming the encryption key $k$ is uniformly distributed.

Prove the one-time pad is perfectly secure

## 2. Perfectly secure ciphers have long keys

Prove that if a cipher is perfectly secure then the key space must be at least as large as the message space.

## 3. A non-cryptographic group action

Let 

$$G = \left\{\begin{pmatrix}c_1&c_2\\c_2&c_1\end{pmatrix} \;\middle|\; c_1,c_2\in 𝔽_p,\; c_1 ≠ ±c_2 \right\}
\qquad\text{and}\qquad
X = \left\{\begin{pmatrix}c_1\\c_2\end{pmatrix} \;\middle|\; c_1,c_2\in 𝔽_p\right\}.$$

1. Prove that $G$ is an abelian group with respect to matrix multiplication.
2. Prove that matrix-vector multiplication $A·v = w$ is a group action of $G$ on $X$.
3. Is the group action regular? Is there a subset of $X$ on which the action is regular?
4. Prove that this action is not a cryptographic group action.